In [0]:
# Inspect failed workflow_summary task output

import requests
import json

RUN_ID = 1087681002053429
FAILED_TASK_KEY = "workflow_summary"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# 1. Get run details
run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

run_response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if run_response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {run_response.status_code} {run_response.text}")

run = run_response.json()
tasks = run.get("tasks", [])

workflow_task = next(
    (task for task in tasks if task.get("task_key") == FAILED_TASK_KEY and task.get("state", {}).get("result_state") == "FAILED"),
    None
)

if workflow_task is None:
    print("No FAILED workflow_summary task found. Showing all workflow_summary tasks:")
    for task in tasks:
        if task.get("task_key") == FAILED_TASK_KEY:
            print(json.dumps({
                "task_key": task.get("task_key"),
                "run_id": task.get("run_id"),
                "state": task.get("state"),
                "run_page_url": task.get("run_page_url"),
            }, indent=2))
    raise ValueError("Could not identify failed workflow_summary task.")

workflow_run_id = workflow_task.get("run_id")

print("Failed workflow_summary task found")
print("=" * 100)
print(json.dumps({
    "task_key": workflow_task.get("task_key"),
    "run_id": workflow_run_id,
    "state": workflow_task.get("state"),
    "run_page_url": workflow_task.get("run_page_url"),
}, indent=2))

# 2. Pull notebook output/error
get_output_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output"

output_response = requests.get(
    get_output_url,
    headers=headers,
    params={"run_id": workflow_run_id},
    timeout=60,
)

if output_response.status_code != 200:
    raise RuntimeError(
        f"Failed to fetch task output: {output_response.status_code} {output_response.text}"
    )

payload = output_response.json()

print("\nNotebook output")
print("=" * 100)
print(payload.get("notebook_output", {}).get("result", ""))

print("\nError")
print("=" * 100)
print(payload.get("error", ""))

print("\nError trace")
print("=" * 100)
error_trace = payload.get("error_trace", "")
print(error_trace[:12000] if error_trace else "")

print("\nRun state")
print("=" * 100)
print(json.dumps(payload.get("metadata", {}).get("state", {}), indent=2))

In [0]:
# Cell 1B: Find the current SupplySage Lakeflow job by name
# Purpose:
# - The old JOB_ID was not found, so search visible jobs in this workspace
# - Do not modify any job config yet

import requests
import json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

print("Current Databricks API host:")
print(DATABRICKS_HOST)

jobs_list_url = f"{DATABRICKS_HOST}/api/2.1/jobs/list"

all_jobs = []
offset = 0
limit = 100

while True:
    response = requests.get(
        jobs_list_url,
        headers=headers,
        params={
            "limit": limit,
            "offset": offset,
            "expand_tasks": "true",
        },
        timeout=60,
    )

    if response.status_code != 200:
        raise RuntimeError(f"Failed to list jobs: {response.status_code} {response.text}")

    payload = response.json()
    jobs = payload.get("jobs", [])
    all_jobs.extend(jobs)

    if len(jobs) < limit:
        break

    offset += limit

print(f"\nTotal visible jobs: {len(all_jobs)}")

matching_jobs = []

for job in all_jobs:
    job_id = job.get("job_id")
    settings = job.get("settings", {})
    name = settings.get("name", "")

    name_lower = name.lower()

    if (
        "supplysage" in name_lower
        or "supply sage" in name_lower
        or "lakeflow" in name_lower
        or "daily refresh" in name_lower
    ):
        matching_jobs.append(job)

print("\nMatching jobs:")
print("=" * 100)

if not matching_jobs:
    print("No matching jobs found by name.")
    print("\nShowing first 20 visible jobs so we can identify the correct one:")
    for job in all_jobs[:20]:
        settings = job.get("settings", {})
        print({
            "job_id": job.get("job_id"),
            "name": settings.get("name"),
            "task_count": len(settings.get("tasks", [])),
        })
else:
    for job in matching_jobs:
        settings = job.get("settings", {})
        tasks = settings.get("tasks", [])

        print({
            "job_id": job.get("job_id"),
            "name": settings.get("name"),
            "task_count": len(tasks),
            "max_concurrent_runs": settings.get("max_concurrent_runs"),
        })

        print("Tasks:")
        for task in tasks:
            print(" -", task.get("task_key"), "=>", task.get("notebook_task", {}).get("notebook_path"))

        print("-" * 100)

print("\nNext step:")
print("Copy the correct job_id from the matching job above. Then we will inspect that job config before adding Notebook 34.")

In [0]:
# Cell 2: Inspect current task config and confirm Notebook 34 path
# Purpose:
# - Use the correct current Job ID
# - Confirm Notebook 34 exists in the workspace
# - Inspect task dependency/compute structure before modifying the job
# - Do NOT update anything yet

import requests
import json

JOB_ID = 522140299855346
NOTEBOOK_34_PATH = "/Users/vigya.awasthi@tamu.edu/supplysage-ai/34_ui_contract_views"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# ------------------------------------------------------------
# 1. Confirm Notebook 34 exists
# ------------------------------------------------------------

workspace_status_url = f"{DATABRICKS_HOST}/api/2.0/workspace/get-status"

path_response = requests.get(
    workspace_status_url,
    headers=headers,
    params={"path": NOTEBOOK_34_PATH},
    timeout=60,
)

print("Notebook 34 path check")
print("=" * 100)
print("Path:", NOTEBOOK_34_PATH)
print("Status code:", path_response.status_code)

if path_response.status_code == 200:
    print("Notebook 34 exists.")
    print(json.dumps(path_response.json(), indent=2))
else:
    print("Notebook 34 was not found at this path.")
    print(path_response.text)
    raise ValueError("Fix NOTEBOOK_34_PATH before updating Lakeflow job.")

# ------------------------------------------------------------
# 2. Fetch current job config
# ------------------------------------------------------------

job_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/get"

job_response = requests.get(
    job_get_url,
    headers=headers,
    params={"job_id": JOB_ID},
    timeout=60,
)

if job_response.status_code != 200:
    raise RuntimeError(f"Failed to fetch job config: {job_response.status_code} {job_response.text}")

job_config = job_response.json()
settings = job_config["settings"]
tasks = settings.get("tasks", [])

print("\nJob config summary")
print("=" * 100)
print("Job ID:", JOB_ID)
print("Job name:", settings.get("name"))
print("Task count:", len(tasks))
print("Max concurrent runs:", settings.get("max_concurrent_runs"))

# ------------------------------------------------------------
# 3. Check whether Notebook 34 task already exists
# ------------------------------------------------------------

existing_ui_tasks = []

for task in tasks:
    task_key = task.get("task_key", "")
    notebook_path = task.get("notebook_task", {}).get("notebook_path", "")

    if (
        "34_ui_contract_views" in task_key
        or "ui_contract" in task_key
        or notebook_path == NOTEBOOK_34_PATH
        or "34_ui_contract_views" in notebook_path
    ):
        existing_ui_tasks.append(task)

print("\nExisting UI contract tasks")
print("=" * 100)

if existing_ui_tasks:
    for task in existing_ui_tasks:
        print(json.dumps(task, indent=2))
    raise ValueError("A UI contract task already exists. Inspect before adding another.")
else:
    print("No existing Notebook 34/UI contract task found. Safe to prepare a new task.")

# ------------------------------------------------------------
# 4. Inspect likely upstream task configs
# ------------------------------------------------------------

tasks_to_inspect = [
    "alerting_notification_routing",
    "agent_runtime_smoke_test",
    "agent_monitoring_eval",
    "serving_health_check",
    "workflow_summary",
]

print("\nSelected upstream/final task configs")
print("=" * 100)

task_by_key = {task.get("task_key"): task for task in tasks}

for task_key in tasks_to_inspect:
    task = task_by_key.get(task_key)

    print(f"\nTASK: {task_key}")
    print("-" * 100)

    if not task:
        print("Not found.")
        continue

    print(json.dumps(task, indent=2))

# ------------------------------------------------------------
# 5. Print dependency map compactly
# ------------------------------------------------------------

print("\nCompact dependency map")
print("=" * 100)

for task in tasks:
    task_key = task.get("task_key")
    notebook_path = task.get("notebook_task", {}).get("notebook_path")
    depends_on = [d.get("task_key") for d in task.get("depends_on", [])]

    print({
        "task_key": task_key,
        "depends_on": depends_on,
        "notebook_path": notebook_path,
    })

print("\nNext step:")
print("After reviewing this output, we will add one new task:")
print("task_key = ui_contract_refresh_34_ui_contract_views")
print("notebook_path =", NOTEBOOK_34_PATH)
print("depends_on = alerting_notification_routing")

In [0]:
# Cell 3: Add Notebook 34 to the Lakeflow job as the final task
# Purpose:
# - Append 34_ui_contract_views to the existing Lakeflow job
# - Do not rebuild the job from scratch
# - Do not modify existing tasks unless needed
# - Dry-run first, then set APPLY_UPDATE=True to apply

import requests
import json
from copy import deepcopy

JOB_ID = 522140299855346

NOTEBOOK_34_TASK_KEY = "ui_contract_refresh_34_ui_contract_views"
NOTEBOOK_34_PATH = "/Users/vigya.awasthi@tamu.edu/supplysage-ai/34_ui_contract_views"

# Dry run first.
# After inspecting the preview output, set this to True and rerun this cell.
APPLY_UPDATE = True

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# -------------------------------------------------------------------
# 1. Fetch current job settings
# -------------------------------------------------------------------

job_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/get"

job_response = requests.get(
    job_get_url,
    headers=headers,
    params={"job_id": JOB_ID},
    timeout=60,
)

if job_response.status_code != 200:
    raise RuntimeError(
        f"Failed to fetch job config: {job_response.status_code} {job_response.text}"
    )

job_config = job_response.json()
settings = job_config["settings"]
current_tasks = settings.get("tasks", [])

print("Current job")
print("=" * 100)
print("Job ID:", JOB_ID)
print("Job name:", settings.get("name"))
print("Current task count:", len(current_tasks))

# -------------------------------------------------------------------
# 2. Safety checks
# -------------------------------------------------------------------

task_keys = [task.get("task_key") for task in current_tasks]

if NOTEBOOK_34_TASK_KEY in task_keys:
    raise ValueError(f"Task already exists: {NOTEBOOK_34_TASK_KEY}")

if "workflow_summary" not in task_keys:
    raise ValueError("Expected upstream task workflow_summary was not found.")

# Confirm Notebook 34 path exists
workspace_status_url = f"{DATABRICKS_HOST}/api/2.0/workspace/get-status"

path_response = requests.get(
    workspace_status_url,
    headers=headers,
    params={"path": NOTEBOOK_34_PATH},
    timeout=60,
)

if path_response.status_code != 200:
    raise ValueError(
        f"Notebook 34 path not found: {NOTEBOOK_34_PATH}\n{path_response.text}"
    )

print("Notebook 34 path exists:", NOTEBOOK_34_PATH)

# -------------------------------------------------------------------
# 3. Build the new task
# -------------------------------------------------------------------

new_task = {
    "task_key": NOTEBOOK_34_TASK_KEY,
    "depends_on": [
        {
            "task_key": "workflow_summary"
        }
    ],
    "run_if": "ALL_SUCCESS",
    "notebook_task": {
        "notebook_path": NOTEBOOK_34_PATH,
        "base_parameters": {
            "project_name": "SupplySage AI",
            "workflow_name": "supplysage_lakeflow_daily_refresh",
            "workflow_version": "v1_lakeflow_jobs_orchestration",
            "run_mode": "production",
            "refresh_scope": "daily",
            "ui_contract_mode": "refresh_and_validate"
        },
        "source": "WORKSPACE"
    },
    "max_retries": 1,
    "min_retry_interval_millis": 300000,
    "retry_on_timeout": False,
    "timeout_seconds": 1800,
    "email_notifications": {},
    "description": "Refresh and validate SupplySage frontend-facing UI contract views and route registry for the Next.js app.",
    "disabled": False
}

# -------------------------------------------------------------------
# 4. Create updated settings payload
# -------------------------------------------------------------------

new_settings = deepcopy(settings)
new_settings["tasks"] = current_tasks + [new_task]

print("\nPlanned new task")
print("=" * 100)
print(json.dumps(new_task, indent=2))

print("\nUpdated job summary preview")
print("=" * 100)
print("Old task count:", len(current_tasks))
print("New task count:", len(new_settings["tasks"]))
print("New final task:", NOTEBOOK_34_TASK_KEY)
print("Depends on:", [d["task_key"] for d in new_task["depends_on"]])

print("\nFinal task order preview")
print("=" * 100)
for i, task in enumerate(new_settings["tasks"], start=1):
    print(f"{i}. {task.get('task_key')}")

# -------------------------------------------------------------------
# 5. Apply update only if explicitly enabled
# -------------------------------------------------------------------

if not APPLY_UPDATE:
    print("\nDRY RUN ONLY")
    print("No job changes were applied.")
    print("If the preview looks correct, set APPLY_UPDATE = True and rerun this cell.")
else:
    reset_url = f"{DATABRICKS_HOST}/api/2.1/jobs/reset"

    reset_payload = {
        "job_id": JOB_ID,
        "new_settings": new_settings,
    }

    reset_response = requests.post(
        reset_url,
        headers=headers,
        data=json.dumps(reset_payload),
        timeout=120,
    )

    if reset_response.status_code != 200:
        raise RuntimeError(
            f"Failed to reset/update job: {reset_response.status_code} {reset_response.text}"
        )

    print("\n✅ Job updated successfully.")
    print(f"Added task: {NOTEBOOK_34_TASK_KEY}")
    print(f"Notebook path: {NOTEBOOK_34_PATH}")
    print("Next step: inspect the job again and trigger a test run.")

In [0]:
# Cell 4: Verify Notebook 34 was added to the Lakeflow job

import requests
import json

JOB_ID = 522140299855346
EXPECTED_TASK_KEY = "ui_contract_refresh_34_ui_contract_views"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

job_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/get"

response = requests.get(
    job_get_url,
    headers=headers,
    params={"job_id": JOB_ID},
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch job config: {response.status_code} {response.text}")

settings = response.json()["settings"]
tasks = settings.get("tasks", [])

matching_tasks = [
    task for task in tasks
    if task.get("task_key") == EXPECTED_TASK_KEY
]

print("Job name:", settings.get("name"))
print("Task count:", len(tasks))

if not matching_tasks:
    raise ValueError(f"Task not found: {EXPECTED_TASK_KEY}")

task = matching_tasks[0]

print("\n✅ Notebook 34 task found.")
print(json.dumps(task, indent=2))

depends_on = [d.get("task_key") for d in task.get("depends_on", [])]

if depends_on != ["workflow_summary"]:
    raise ValueError(f"Unexpected dependency: {depends_on}")

print("\n✅ Dependency is correct:")
print(f"{EXPECTED_TASK_KEY} depends on workflow_summary")

In [0]:
# Cell 5: Trigger one Lakeflow test run

import requests
import json

JOB_ID = 522140299855346

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

run_now_url = f"{DATABRICKS_HOST}/api/2.1/jobs/run-now"

payload = {
    "job_id": JOB_ID
}

response = requests.post(
    run_now_url,
    headers=headers,
    data=json.dumps(payload),
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to trigger job: {response.status_code} {response.text}")

run_info = response.json()

print("✅ Lakeflow job triggered.")
print(json.dumps(run_info, indent=2))
print("\nOpen the job run in Databricks Workflows and confirm:")
print("1. alerting_notification_routing succeeds")
print("2. workflow_summary succeeds")
print("3. ui_contract_refresh_34_ui_contract_views succeeds")

In [0]:
# Cell 7: Focused monitor for final Lakeflow tasks

import requests
import json

RUN_ID = 23761521294949

IMPORTANT_TASKS = [
    "bronze_external_ingestion",
    "serving_health_check",
    "alerting_notification_routing",
    "workflow_summary",
    "ui_contract_refresh_34_ui_contract_views",
]

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {response.status_code} {response.text}")

run = response.json()

overall_state = run.get("state", {})

print("Overall run status")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Life cycle state:", overall_state.get("life_cycle_state"))
print("Result state:", overall_state.get("result_state"))
print("Message:", overall_state.get("state_message"))

print("\nImportant task statuses")
print("=" * 100)

tasks = run.get("tasks", [])

for task_key in IMPORTANT_TASKS:
    task = next((t for t in tasks if t.get("task_key") == task_key), None)

    if not task:
        print(f"{task_key}: NOT FOUND")
        continue

    state = task.get("state", {})

    print({
        "task_key": task_key,
        "run_id": task.get("run_id"),
        "life_cycle_state": state.get("life_cycle_state"),
        "result_state": state.get("result_state"),
        "message": state.get("state_message"),
    })

print("\nInterpretation:")
print("- BLOCKED means waiting for upstream tasks.")
print("- RUNNING means currently executing.")
print("- TERMINATED + SUCCESS means completed successfully.")
print("- TERMINATED + FAILED means open that task run and inspect the error.")

In [0]:
# Cell 8: Monitor active, failed, and final Lakeflow tasks

import requests
import json

RUN_ID = 23761521294949
UI_TASK_KEY = "ui_contract_refresh_34_ui_contract_views"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {response.status_code} {response.text}")

run = response.json()
tasks = run.get("tasks", [])
overall_state = run.get("state", {})

print("Overall run status")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Life cycle state:", overall_state.get("life_cycle_state"))
print("Result state:", overall_state.get("result_state"))
print("Message:", overall_state.get("state_message"))

active_tasks = []
failed_tasks = []
successful_tasks = []
blocked_tasks = []

for task in tasks:
    state = task.get("state", {})
    life_cycle_state = state.get("life_cycle_state")
    result_state = state.get("result_state")

    row = {
        "task_key": task.get("task_key"),
        "run_id": task.get("run_id"),
        "life_cycle_state": life_cycle_state,
        "result_state": result_state,
        "message": state.get("state_message"),
    }

    if life_cycle_state in ["RUNNING", "PENDING", "QUEUED"]:
        active_tasks.append(row)

    if result_state in ["FAILED", "TIMEDOUT", "CANCELED"]:
        failed_tasks.append(row)

    if life_cycle_state == "TERMINATED" and result_state == "SUCCESS":
        successful_tasks.append(row)

    if life_cycle_state == "BLOCKED":
        blocked_tasks.append(row)

print("\nProgress summary")
print("=" * 100)
print("Successful tasks:", len(successful_tasks))
print("Active tasks:", len(active_tasks))
print("Blocked/waiting tasks:", len(blocked_tasks))
print("Failed tasks:", len(failed_tasks))
print("Total tasks:", len(tasks))

print("\nCurrently active tasks")
print("=" * 100)
if active_tasks:
    for row in active_tasks:
        print(row)
else:
    print("No active tasks right now.")

print("\nFailed tasks")
print("=" * 100)
if failed_tasks:
    for row in failed_tasks:
        print(row)
else:
    print("No failed tasks.")

print("\nNotebook 34 status")
print("=" * 100)
ui_task = next((t for t in tasks if t.get("task_key") == UI_TASK_KEY), None)

if ui_task:
    state = ui_task.get("state", {})
    print({
        "task_key": ui_task.get("task_key"),
        "run_id": ui_task.get("run_id"),
        "life_cycle_state": state.get("life_cycle_state"),
        "result_state": state.get("result_state"),
        "message": state.get("state_message"),
        "run_page_url": ui_task.get("run_page_url"),
    })
else:
    print("Notebook 34 task not found in this run.")

print("\nNext action:")
if failed_tasks:
    print("A task failed. Open the failed task run and inspect the error.")
elif overall_state.get("life_cycle_state") == "TERMINATED":
    print("Run finished. If result is SUCCESS, validate the UI contracts.")
else:
    print("Run is still in progress. Wait a few minutes and rerun this cell.")

In [0]:
# Cell 9: Inspect failed Notebook 18 task output
# Failed task:
# gold_marts_refresh_03_18_gold_supplier_sku_dependency_mart
#
# Purpose:
# - Pull the actual Databricks notebook error/output for the failed task attempts
# - Identify the exact code/table/column issue before patching

import requests
import json

FAILED_TASK_RUN_IDS = [
    527153082414911,
    175535743746746,
]

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

get_output_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output"

for task_run_id in FAILED_TASK_RUN_IDS:
    print("\n" + "=" * 120)
    print(f"Inspecting failed task run ID: {task_run_id}")
    print("=" * 120)

    response = requests.get(
        get_output_url,
        headers=headers,
        params={"run_id": task_run_id},
        timeout=60,
    )

    print("Status code:", response.status_code)

    if response.status_code != 200:
        print(response.text)
        continue

    payload = response.json()

    # Print high-level metadata keys
    print("\nPayload keys:")
    print(payload.keys())

    # Print notebook output if available
    notebook_output = payload.get("notebook_output", {})
    print("\nNotebook output:")
    print(notebook_output.get("result", ""))

    # Print direct error if available
    error = payload.get("error")
    if error:
        print("\nError:")
        print(error)

    # Print error trace if available
    error_trace = payload.get("error_trace")
    if error_trace:
        print("\nError trace:")
        print(error_trace[:5000])

    # Print metadata if useful
    metadata = payload.get("metadata", {})
    state = metadata.get("state", {})
    print("\nRun state:")
    print(json.dumps(state, indent=2))

    print("\nRun page URL:")
    print(metadata.get("run_page_url"))

In [0]:
# Monitor current Lakeflow run after patching Notebook 18

import requests
import json

RUN_ID = 1087681002053429

WATCH_TASKS = [
    "bronze_external_ingestion",
    "gold_marts_refresh_03_18_gold_supplier_sku_dependency_mart",
    "serving_health_check",
    "alerting_notification_routing",
    "workflow_summary",
    "ui_contract_refresh_34_ui_contract_views",
]

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {response.status_code} {response.text}")

run = response.json()
tasks = run.get("tasks", [])
overall_state = run.get("state", {})

print("Overall run status")
print("=" * 100)
print("Run ID:", RUN_ID)
print("Life cycle state:", overall_state.get("life_cycle_state"))
print("Result state:", overall_state.get("result_state"))
print("Message:", overall_state.get("state_message"))

successful = []
active = []
failed = []
blocked = []
skipped = []

for task in tasks:
    state = task.get("state", {})
    life_cycle_state = state.get("life_cycle_state")
    result_state = state.get("result_state")

    row = {
        "task_key": task.get("task_key"),
        "run_id": task.get("run_id"),
        "life_cycle_state": life_cycle_state,
        "result_state": result_state,
        "message": state.get("state_message"),
        "run_page_url": task.get("run_page_url"),
    }

    if life_cycle_state in ["RUNNING", "PENDING", "QUEUED"]:
        active.append(row)
    elif life_cycle_state == "BLOCKED":
        blocked.append(row)
    elif life_cycle_state == "SKIPPED":
        skipped.append(row)

    if result_state == "SUCCESS":
        successful.append(row)
    elif result_state in ["FAILED", "TIMEDOUT", "CANCELED", "UPSTREAM_FAILED"]:
        failed.append(row)

print("\nProgress summary")
print("=" * 100)
print("Successful:", len(successful))
print("Active:", len(active))
print("Blocked:", len(blocked))
print("Skipped:", len(skipped))
print("Failed:", len(failed))
print("Total tasks:", len(tasks))

print("\nCurrently active tasks")
print("=" * 100)
if active:
    for row in active:
        print(row)
else:
    print("No active tasks.")

print("\nWatched task statuses")
print("=" * 100)

for task_key in WATCH_TASKS:
    task = next((t for t in tasks if t.get("task_key") == task_key), None)

    if not task:
        print(f"{task_key}: NOT FOUND")
        continue

    state = task.get("state", {})

    print({
        "task_key": task_key,
        "run_id": task.get("run_id"),
        "life_cycle_state": state.get("life_cycle_state"),
        "result_state": state.get("result_state"),
        "message": state.get("state_message"),
    })

print("\nFailed tasks")
print("=" * 100)
if failed:
    for row in failed:
        print(row)
else:
    print("No failed tasks.")

print("\nNext action")
print("=" * 100)
if failed:
    print("A task failed. Inspect the failed task output.")
elif overall_state.get("life_cycle_state") == "TERMINATED" and overall_state.get("result_state") == "SUCCESS":
    print("✅ Full Lakeflow run succeeded. Now validate UI contracts.")
else:
    print("Run is still in progress. Wait a few minutes and rerun this cell.")

In [0]:
# Inspect failed task output for the current Lakeflow run
# Current run ID from screenshot: 1087681002053429

import requests
import json

RUN_ID = 1087681002053429

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# ------------------------------------------------------------
# 1. Get run details and find failed tasks
# ------------------------------------------------------------

run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

run_response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if run_response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {run_response.status_code} {run_response.text}")

run = run_response.json()
tasks = run.get("tasks", [])

failed_tasks = []

for task in tasks:
    state = task.get("state", {})
    result_state = state.get("result_state")
    life_cycle_state = state.get("life_cycle_state")

    if result_state in ["FAILED", "TIMEDOUT", "CANCELED"]:
        failed_tasks.append({
            "task_key": task.get("task_key"),
            "run_id": task.get("run_id"),
            "life_cycle_state": life_cycle_state,
            "result_state": result_state,
            "message": state.get("state_message"),
            "run_page_url": task.get("run_page_url"),
        })

print("Failed tasks")
print("=" * 100)

if not failed_tasks:
    print("No failed tasks found.")
else:
    for task in failed_tasks:
        print(json.dumps(task, indent=2))

# ------------------------------------------------------------
# 2. Pull notebook output/error for each failed task
# ------------------------------------------------------------

get_output_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output"

for task in failed_tasks:
    task_run_id = task["run_id"]
    task_key = task["task_key"]

    print("\n" + "=" * 120)
    print(f"Inspecting failed task: {task_key}")
    print(f"Task run ID: {task_run_id}")
    print("=" * 120)

    output_response = requests.get(
        get_output_url,
        headers=headers,
        params={"run_id": task_run_id},
        timeout=60,
    )

    print("Status code:", output_response.status_code)

    if output_response.status_code != 200:
        print(output_response.text)
        continue

    payload = output_response.json()

    notebook_output = payload.get("notebook_output", {})
    error = payload.get("error")
    error_trace = payload.get("error_trace")

    print("\nNotebook output:")
    print(notebook_output.get("result", ""))

    print("\nError:")
    print(error if error else "")

    print("\nError trace:")
    if error_trace:
        print(error_trace[:8000])
    else:
        print("")

    print("\nRun page URL:")
    print(task.get("run_page_url"))

In [0]:
# Inspect failed alerting task output from the current Lakeflow run

import requests
import json

RUN_ID = 1087681002053429
FAILED_TASK_KEY = "alerting_notification_routing"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# 1. Get run details
run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

run_response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if run_response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {run_response.status_code} {run_response.text}")

run = run_response.json()
tasks = run.get("tasks", [])

alerting_task = next(
    (task for task in tasks if task.get("task_key") == FAILED_TASK_KEY),
    None
)

if alerting_task is None:
    raise ValueError(f"Task not found: {FAILED_TASK_KEY}")

alerting_run_id = alerting_task.get("run_id")

print("Alerting task found")
print("=" * 100)
print(json.dumps({
    "task_key": alerting_task.get("task_key"),
    "run_id": alerting_run_id,
    "state": alerting_task.get("state"),
    "run_page_url": alerting_task.get("run_page_url"),
}, indent=2))

# 2. Pull notebook output/error
get_output_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output"

output_response = requests.get(
    get_output_url,
    headers=headers,
    params={"run_id": alerting_run_id},
    timeout=60,
)

if output_response.status_code != 200:
    raise RuntimeError(
        f"Failed to fetch task output: {output_response.status_code} {output_response.text}"
    )

payload = output_response.json()

print("\nNotebook output")
print("=" * 100)
print(payload.get("notebook_output", {}).get("result", ""))

print("\nError")
print("=" * 100)
print(payload.get("error", ""))

print("\nError trace")
print("=" * 100)
error_trace = payload.get("error_trace", "")
print(error_trace[:8000] if error_trace else "")

In [0]:
# Find the real upstream blocker for alerting_notification_routing

import requests
import json

RUN_ID = 1087681002053429
TARGET_TASK_KEY = "alerting_notification_routing"

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

run_get_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get"

response = requests.get(
    run_get_url,
    headers=headers,
    params={"run_id": RUN_ID},
    timeout=60,
)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch run: {response.status_code} {response.text}")

run = response.json()
tasks = run.get("tasks", [])

task_by_key = {task.get("task_key"): task for task in tasks}

print("Overall run status")
print("=" * 100)
print(json.dumps(run.get("state", {}), indent=2))

print("\nAll non-success tasks")
print("=" * 100)

non_success = []

for task in tasks:
    state = task.get("state", {})
    life_cycle_state = state.get("life_cycle_state")
    result_state = state.get("result_state")

    if not (life_cycle_state == "TERMINATED" and result_state == "SUCCESS"):
        row = {
            "task_key": task.get("task_key"),
            "run_id": task.get("run_id"),
            "life_cycle_state": life_cycle_state,
            "result_state": result_state,
            "message": state.get("state_message"),
            "depends_on": [d.get("task_key") for d in task.get("depends_on", [])],
            "run_page_url": task.get("run_page_url"),
        }
        non_success.append(row)
        print(json.dumps(row, indent=2))

print("\nDependency chain for alerting_notification_routing")
print("=" * 100)

def print_dependency_chain(task_key, level=0, visited=None):
    if visited is None:
        visited = set()

    indent = "  " * level

    if task_key in visited:
        print(f"{indent}- {task_key}: already visited")
        return

    visited.add(task_key)

    task = task_by_key.get(task_key)

    if not task:
        print(f"{indent}- {task_key}: NOT FOUND")
        return

    state = task.get("state", {})
    depends_on = [d.get("task_key") for d in task.get("depends_on", [])]

    print(
        f"{indent}- {task_key}: "
        f"{state.get('life_cycle_state')} / {state.get('result_state')} "
        f"| message={state.get('state_message')}"
    )

    for upstream_key in depends_on:
        print_dependency_chain(upstream_key, level + 1, visited)

print_dependency_chain(TARGET_TASK_KEY)

print("\nNext action")
print("=" * 100)

actual_failed = [
    row for row in non_success
    if row["result_state"] in ["FAILED", "TIMEDOUT", "CANCELED"]
]

if actual_failed:
    print("Inspect the actual failed task output for these tasks:")
    for row in actual_failed:
        print(f"- {row['task_key']} | run_id={row['run_id']}")
else:
    print("No actual FAILED task found. This may be a repair-run dependency selection issue.")
    print("In that case, trigger a new full run or repair from the earliest skipped dependency with downstream tasks included.")

In [0]:
# Inspect the actual failed alerting task attempt
# Current real blocker:
# alerting_notification_routing | run_id=138471561807005

import requests
import json

ALERTING_FAILED_RUN_ID = 138471561807005

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

get_output_url = f"{DATABRICKS_HOST}/api/2.1/jobs/runs/get-output"

response = requests.get(
    get_output_url,
    headers=headers,
    params={"run_id": ALERTING_FAILED_RUN_ID},
    timeout=60,
)

print("Status code:", response.status_code)

if response.status_code != 200:
    raise RuntimeError(f"Failed to fetch task output: {response.status_code} {response.text}")

payload = response.json()

print("\nPayload keys")
print("=" * 100)
print(payload.keys())

print("\nNotebook output")
print("=" * 100)
print(payload.get("notebook_output", {}).get("result", ""))

print("\nError")
print("=" * 100)
print(payload.get("error", ""))

print("\nError trace")
print("=" * 100)
error_trace = payload.get("error_trace", "")
print(error_trace[:12000] if error_trace else "")

metadata = payload.get("metadata", {})
print("\nRun state")
print("=" * 100)
print(json.dumps(metadata.get("state", {}), indent=2))

print("\nRun page URL")
print("=" * 100)
print(metadata.get("run_page_url"))

In [0]:
# Verify current Lakeflow job still has Notebook 34

import requests
import json

JOB_ID = 522140299855346

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

response = requests.get(
    f"{DATABRICKS_HOST}/api/2.1/jobs/get",
    headers=headers,
    params={"job_id": JOB_ID},
    timeout=60,
)

response.raise_for_status()

settings = response.json()["settings"]
tasks = settings.get("tasks", [])

print("Job name:", settings.get("name"))
print("Task count:", len(tasks))

for task in tasks:
    if task.get("task_key") in [
        "workflow_summary",
        "ui_contract_refresh_34_ui_contract_views",
    ]:
        print(json.dumps({
            "task_key": task.get("task_key"),
            "notebook_path": task.get("notebook_task", {}).get("notebook_path"),
            "depends_on": [d.get("task_key") for d in task.get("depends_on", [])],
        }, indent=2))

assert any(
    t.get("task_key") == "ui_contract_refresh_34_ui_contract_views"
    for t in tasks
), "Notebook 34 task is missing."

print("✅ Notebook 34 task is present.")

In [0]:
# Restore Lakeflow job structure after Notebook 30 reset
# Purpose:
# - Ensure alerting_notification_routing exists
# - Ensure workflow_summary waits for both serving_health_check and alerting_notification_routing
# - Ensure Notebook 34 UI contract task exists after workflow_summary
# - Dry run first, then set APPLY_UPDATE=True

import requests
import json
from copy import deepcopy

JOB_ID = 522140299855346

NOTEBOOK_32_TASK_KEY = "alerting_notification_routing"
NOTEBOOK_32_PATH = "/Users/vigya.awasthi@tamu.edu/supplysage-ai/32_alerting_notification_routing"

WORKFLOW_SUMMARY_TASK_KEY = "workflow_summary"

NOTEBOOK_34_TASK_KEY = "ui_contract_refresh_34_ui_contract_views"
NOTEBOOK_34_PATH = "/Users/vigya.awasthi@tamu.edu/supplysage-ai/34_ui_contract_views"

APPLY_UPDATE = True  # change to True only after preview looks correct

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DATABRICKS_HOST = ctx.apiUrl().get()
DATABRICKS_TOKEN = ctx.apiToken().get()

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# ------------------------------------------------------------
# 1. Fetch current job config
# ------------------------------------------------------------

job_response = requests.get(
    f"{DATABRICKS_HOST}/api/2.1/jobs/get",
    headers=headers,
    params={"job_id": JOB_ID},
    timeout=60,
)

job_response.raise_for_status()

job_config = job_response.json()
settings = job_config["settings"]
current_tasks = settings.get("tasks", [])

print("Current job")
print("=" * 100)
print("Job name:", settings.get("name"))
print("Current task count:", len(current_tasks))

task_by_key = {task.get("task_key"): task for task in current_tasks}

# ------------------------------------------------------------
# 2. Confirm notebook paths exist
# ------------------------------------------------------------

def assert_workspace_path_exists(path: str):
    response = requests.get(
        f"{DATABRICKS_HOST}/api/2.0/workspace/get-status",
        headers=headers,
        params={"path": path},
        timeout=60,
    )

    if response.status_code != 200:
        raise ValueError(f"Workspace path not found: {path}\n{response.text}")

    print(f"✅ Path exists: {path}")

assert_workspace_path_exists(NOTEBOOK_32_PATH)
assert_workspace_path_exists(NOTEBOOK_34_PATH)

# ------------------------------------------------------------
# 3. Prepare updated task list
# ------------------------------------------------------------

new_tasks = deepcopy(current_tasks)
new_task_by_key = {task.get("task_key"): task for task in new_tasks}

# 3A. Restore alerting task if missing
if NOTEBOOK_32_TASK_KEY not in new_task_by_key:
    alerting_task = {
        "task_key": NOTEBOOK_32_TASK_KEY,
        "depends_on": [
            {
                "task_key": "serving_health_check"
            }
        ],
        "run_if": "ALL_SUCCESS",
        "notebook_task": {
            "notebook_path": NOTEBOOK_32_PATH,
            "base_parameters": {
                "run_mode": "production",
                "alerting_mode": "scheduled_lakeflow",
                "send_enabled_default": "false"
            },
            "source": "WORKSPACE"
        },
        "timeout_seconds": 0,
        "email_notifications": {},
        "description": "Generate SupplySage alert events, route alerts through Delta-backed config, create alert inbox records, and log delivery attempts.",
        "disabled": False
    }

    new_tasks.append(alerting_task)
    new_task_by_key[NOTEBOOK_32_TASK_KEY] = alerting_task
    print(f"Will add missing task: {NOTEBOOK_32_TASK_KEY}")
else:
    print(f"Task already exists: {NOTEBOOK_32_TASK_KEY}")

# 3B. Ensure workflow_summary depends on both serving health and alerting
if WORKFLOW_SUMMARY_TASK_KEY not in new_task_by_key:
    raise ValueError("workflow_summary task is missing. Stop and inspect job manually.")

workflow_task = new_task_by_key[WORKFLOW_SUMMARY_TASK_KEY]

workflow_task["depends_on"] = [
    {
        "task_key": "serving_health_check"
    },
    {
        "task_key": NOTEBOOK_32_TASK_KEY
    }
]

workflow_task["run_if"] = "ALL_SUCCESS"

print("Will set workflow_summary depends_on to:")
print([d["task_key"] for d in workflow_task["depends_on"]])

# 3C. Restore Notebook 34 UI contract task if missing
if NOTEBOOK_34_TASK_KEY not in new_task_by_key:
    ui_contract_task = {
        "task_key": NOTEBOOK_34_TASK_KEY,
        "depends_on": [
            {
                "task_key": WORKFLOW_SUMMARY_TASK_KEY
            }
        ],
        "run_if": "ALL_SUCCESS",
        "notebook_task": {
            "notebook_path": NOTEBOOK_34_PATH,
            "base_parameters": {
                "project_name": "SupplySage AI",
                "workflow_name": "supplysage_lakeflow_daily_refresh",
                "workflow_version": "v1_lakeflow_jobs_orchestration",
                "run_mode": "production",
                "refresh_scope": "daily",
                "ui_contract_mode": "refresh_and_validate"
            },
            "source": "WORKSPACE"
        },
        "max_retries": 1,
        "min_retry_interval_millis": 300000,
        "retry_on_timeout": False,
        "timeout_seconds": 1800,
        "email_notifications": {},
        "description": "Refresh and validate SupplySage frontend-facing UI contract views and route registry for the Next.js app.",
        "disabled": False
    }

    new_tasks.append(ui_contract_task)
    new_task_by_key[NOTEBOOK_34_TASK_KEY] = ui_contract_task
    print(f"Will add missing task: {NOTEBOOK_34_TASK_KEY}")
else:
    print(f"Task already exists: {NOTEBOOK_34_TASK_KEY}")

# ------------------------------------------------------------
# 4. Build updated settings
# ------------------------------------------------------------

new_settings = deepcopy(settings)
new_settings["tasks"] = new_tasks

print("\nUpdate preview")
print("=" * 100)
print("Old task count:", len(current_tasks))
print("New task count:", len(new_tasks))

for task_key in [
    NOTEBOOK_32_TASK_KEY,
    WORKFLOW_SUMMARY_TASK_KEY,
    NOTEBOOK_34_TASK_KEY,
]:
    task = new_task_by_key.get(task_key)
    print("\n", task_key)
    print(json.dumps({
        "task_key": task.get("task_key"),
        "notebook_path": task.get("notebook_task", {}).get("notebook_path"),
        "depends_on": [d.get("task_key") for d in task.get("depends_on", [])],
    }, indent=2))

# ------------------------------------------------------------
# 5. Apply update only if explicitly enabled
# ------------------------------------------------------------

if not APPLY_UPDATE:
    print("\nDRY RUN ONLY")
    print("No changes applied.")
    print("If preview looks correct, set APPLY_UPDATE = True and rerun this cell.")
else:
    reset_response = requests.post(
        f"{DATABRICKS_HOST}/api/2.1/jobs/reset",
        headers=headers,
        data=json.dumps({
            "job_id": JOB_ID,
            "new_settings": new_settings,
        }),
        timeout=120,
    )

    if reset_response.status_code != 200:
        raise RuntimeError(
            f"Failed to update job: {reset_response.status_code} {reset_response.text}"
        )

    print("\n✅ Lakeflow job restored successfully.")
    print("Next: run the verification cell again.")